In [1]:
"""
Dedalus script simulating 2D horizontally-periodic Rayleigh-Benard convection.
This script demonstrates solving a 2D Cartesian initial value problem. It can
be ran serially or in parallel, and uses the built-in analysis framework to save
data snapshots to HDF5 files. The `plot_snapshots.py` script can be used to
produce plots from the saved data. It should take about 5 cpu-minutes to run.

The problem is non-dimensionalized using the box height and freefall time, so
the resulting thermal diffusivity and viscosity are related to the Prandtl
and Rayleigh numbers as:

    kappa = (Rayleigh * Prandtl)**(-1/2)
    nu = (Rayleigh / Prandtl)**(-1/2)

For incompressible hydro with two boundaries, we need two tau terms for each the
velocity and buoyancy. Here we choose to use a first-order formulation, putting
one tau term each on auxiliary first-order gradient variables and the others in
the PDE, and lifting them all to the first derivative basis. This formulation puts
a tau term in the divergence constraint, as required for this geometry.

To run and plot using e.g. 4 processes:
    $ mpiexec -n 4 python3 rayleigh_benard.py
    $ mpiexec -n 4 python3 plot_snapshots.py snapshots/*.h5
"""

import numpy as np
import dedalus.public as d3
import logging
from dedalus.extras.plot_tools import plot_bot_2d
logger = logging.getLogger(__name__)


# Parameters
Lx, Lz = 8, 8
Nx, Nz = 512, 512
Rayleigh = 2e6
Prandtl = 1
dealias = 3/2
stop_sim_time = 0.0002859165868008745 * 50000
timestepper = d3.RK222
max_timestep = 0.125
dtype = np.float64

# Bases
coords = d3.CartesianCoordinates('x', 'z')
dist = d3.Distributor(coords, dtype=dtype)
xbasis = d3.RealFourier(coords['x'], size=Nx, bounds=(0, Lx), dealias=dealias)
zbasis = d3.ChebyshevT(coords['z'], size=Nz, bounds=(0, Lz), dealias=dealias)

# Fields
p = dist.Field(name='p', bases=(xbasis,zbasis))
b = dist.Field(name='b', bases=(xbasis,zbasis))
u = dist.VectorField(coords, name='u', bases=(xbasis,zbasis))
tau_p = dist.Field(name='tau_p')
tau_b1 = dist.Field(name='tau_b1', bases=xbasis)
tau_b2 = dist.Field(name='tau_b2', bases=xbasis)
tau_u1 = dist.VectorField(coords, name='tau_u1', bases=xbasis)
tau_u2 = dist.VectorField(coords, name='tau_u2', bases=xbasis)

# Substitutions
kappa = (Rayleigh * Prandtl)**(-1/2)
nu = (Rayleigh / Prandtl)**(-1/2)
x, z = dist.local_grids(xbasis, zbasis)
ex, ez = coords.unit_vector_fields(dist)
lift_basis = zbasis.derivative_basis(1)
lift = lambda A: d3.Lift(A, lift_basis, -1)
grad_u = d3.grad(u) + ez*lift(tau_u1) # First-order reduction
grad_b = d3.grad(b) + ez*lift(tau_b1) # First-order reduction

# Problem
# First-order form: "div(f)" becomes "trace(grad_f)"
# First-order form: "lap(f)" becomes "div(grad_f)"
problem = d3.IVP([p, b, u, tau_p, tau_b1, tau_b2, tau_u1, tau_u2], namespace=locals())
problem.add_equation("trace(grad_u) + tau_p = 0")
problem.add_equation("dt(b) - kappa*div(grad_b) + lift(tau_b2) = - u@grad(b)")
problem.add_equation("dt(u) - nu*div(grad_u) + grad(p) - b*ez + lift(tau_u2) = - u@grad(u)")
problem.add_equation("b(z=0) = Lz")
problem.add_equation("u(z=0) = 0")
problem.add_equation("b(z=Lz) = 0")
problem.add_equation("u(z=Lz) = 0")
problem.add_equation("integ(p) = 0") # Pressure gauge

# Solver
solver = problem.build_solver(timestepper)
solver.stop_sim_time = stop_sim_time

# Initial conditions
b.fill_random('g', seed=42, distribution='normal', scale=1e-3) # Random noise
b['g'] *= z * (Lz - z) # Damp noise at walls
b['g'] += Lz - z # Add linear background



2025-03-26 03:16:42,091 dedalus 0/1 WARNING :: Threading has not been disabled. This may massively degrade Dedalus performance.
2025-03-26 03:16:42,092 dedalus 0/1 WARNING :: We strongly suggest setting the "OMP_NUM_THREADS" environment variable to "1".
2025-03-26 03:16:42,497 numexpr.utils 0/1 INFO :: NumExpr defaulting to 8 threads.
2025-03-26 03:16:46,718 subsystems 0/1 INFO :: Building subproblem matrices 1/256 (~0%) Elapsed: 0s, Remaining: 39s, Rate: 6.5e+00/s
2025-03-26 03:16:47,597 subsystems 0/1 INFO :: Building subproblem matrices 26/256 (~10%) Elapsed: 1s, Remaining: 9s, Rate: 2.5e+01/s
2025-03-26 03:16:48,467 subsystems 0/1 INFO :: Building subproblem matrices 52/256 (~20%) Elapsed: 2s, Remaining: 7s, Rate: 2.7e+01/s
2025-03-26 03:16:49,333 subsystems 0/1 INFO :: Building subproblem matrices 78/256 (~30%) Elapsed: 3s, Remaining: 6s, Rate: 2.8e+01/s
2025-03-26 03:16:50,234 subsystems 0/1 INFO :: Building subproblem matrices 104/256 (~41%) Elapsed: 4s, Remaining: 5s, Rate: 2.8

In [2]:
# Analysis
snapshots = solver.evaluator.add_file_handler('snapshots', iter = 100, max_writes=5000000000000)
snapshots.add_task(b, name='buoyancy')
snapshots.add_task(-d3.div(d3.skew(u)), name='vorticity')
snapshots.add_task(u, name='velocity')
snapshots.add_task(p, name='pressure')

# CFL
CFL = d3.CFL(solver, initial_dt=max_timestep, cadence=10, safety=0.5, threshold=0.05,
             max_change=1.5, min_change=0.5, max_dt=max_timestep)
CFL.add_velocity(u)

# Flow properties
flow = d3.GlobalFlowProperty(solver, cadence=10)
flow.add_property(np.sqrt(u@u)/nu, name='Re')

# Main loop
startup_iter = 10
min_timestep = 1

try:
    logger.info('Starting main loop')
    while solver.proceed:
        #timestep = CFL.compute_timestep()
        timestep = 0.0002859165868008745 / 2
        solver.step(timestep)
        if timestep < min_timestep:
            min_timestep = timestep
        if (solver.iteration-1) % 10 == 0:
            max_Re = flow.max('Re')
            logger.info('Iteration=%i, Time=%e, dt=%e, max(Re)=%f' %(solver.iteration, solver.sim_time, timestep, max_Re))
            print(CFL)
        '''if (solver.iteration - 1) % 200 == 0:
            vortgrad = -d3.div(d3.skew(u))
            #vortgrad.change_scales(1)  # Ensure consistent grid resolution
            vortgrad_data = vortgrad.evaluate().data
            plot_bot_2d(vortgrad_data, figkw={'figsize':(8,8), 'dpi':100}, title="vorticity grad");'''
except:
    logger.error('Exception raised, triggering end of main loop.')
    raise
finally:
    solver.log_stats()

2025-03-26 03:16:55,473 __main__ 0/1 INFO :: Starting main loop
2025-03-26 03:17:01,735 __main__ 0/1 INFO :: Iteration=1, Time=1.429583e-04, dt=1.429583e-04, max(Re)=0.000000
2025-03-26 03:17:05,831 __main__ 0/1 INFO :: Iteration=11, Time=1.572541e-03, dt=1.429583e-04, max(Re)=0.081435
2025-03-26 03:17:09,907 __main__ 0/1 INFO :: Iteration=21, Time=3.002124e-03, dt=1.429583e-04, max(Re)=0.161237
2025-03-26 03:17:13,975 __main__ 0/1 INFO :: Iteration=31, Time=4.431707e-03, dt=1.429583e-04, max(Re)=0.239387
2025-03-26 03:17:18,137 __main__ 0/1 INFO :: Iteration=41, Time=5.861290e-03, dt=1.429583e-04, max(Re)=0.315867
2025-03-26 03:17:22,305 __main__ 0/1 INFO :: Iteration=51, Time=7.290873e-03, dt=1.429583e-04, max(Re)=0.390715
2025-03-26 03:17:26,481 __main__ 0/1 INFO :: Iteration=61, Time=8.720456e-03, dt=1.429583e-04, max(Re)=0.463992
2025-03-26 03:17:30,587 __main__ 0/1 INFO :: Iteration=71, Time=1.015004e-02, dt=1.429583e-04, max(Re)=0.535757
2025-03-26 03:17:34,844 __main__ 0/1 INFO